# Dynamic Parallel IDK Scheduler



In [1]:
import heapq
import time
from collections import Counter, deque

import numpy as np
from sklearn.ensemble import RandomForestClassifier

RESNET18 = "resnet18"
RESNET34 = "resnet34"
RESNET50 = "resnet50"
RESNET152 = "resnet152"
MODELS = (RESNET18, RESNET34, RESNET50, RESNET152)
HEAVY_MODELS = (RESNET50, RESNET152)

DYNAMIC_CONFIDENCE_THRESHOLD = 0.4
SEQUENTIAL_CONFIDENCE_THRESHOLD = 0.9

HEAVY_ROUTE_LABEL_BY_MODEL = {RESNET50: 0, RESNET152: 1}
HEAVY_MODEL_BY_ROUTE_LABEL = {
    label: model for model, label in HEAVY_ROUTE_LABEL_BY_MODEL.items()
}
SKIP_RESNET34_LABEL = 0
USE_RESNET34_LABEL = 1

## Cache Handling and Features

In [2]:
REQUIRED_CACHE_FIELDS = ("probabilities", "labels", "predictions", "times_ms", "keys")

##### Returns if the cache for the model(s) is valid

In [3]:
def validate_model_caches(caches):
    missing_models = [model for model in MODELS if model not in caches]
    if missing_models:
        raise ValueError(f"Missing model caches: {missing_models}")

    reference_cache = caches[RESNET18]
    sample_count = len(reference_cache["labels"])


    # for model, cache in caches.items():
    #     missing_fields = [field for field in REQUIRED_CACHE_FIELDS if field not in cache]
    #     if missing_fields:
    #         raise ValueError(f"{model} cache is missing: {missing_fields}")
    #     if any(len(cache[field]) != sample_count for field in REQUIRED_CACHE_FIELDS):
    #         raise ValueError(f"{model} cache arrays have different lengths")
    #     if not np.array_equal(cache["labels"], reference_cache["labels"]):
    #         raise ValueError(f"{model} labels do not match {RESNET18}")
    #     if not np.array_equal(cache["keys"], reference_cache["keys"]):
    #         raise ValueError(f"{model} keys do not match {RESNET18}")

    return sample_count

##### Loads the model cache from file path

In [4]:
def load_model_caches(paths_by_model):
    caches = {}
    for model, path in paths_by_model.items():
        with np.load(path) as data:
            caches[model] = {name: data[name] for name in data.files}
    validate_model_caches(caches)
    return caches

##### Returns 'n' sample cache instead of entire dataset

In [5]:
def take_first_samples(caches, sample_count):
    # if not isinstance(sample_count, (int, np.integer)) or sample_count <= 0:
        # raise ValueError("sample_count must be a positive integer")
    sample_count = min(int(sample_count), validate_model_caches(caches))
    sliced_caches = {
        model: {
            field: value[:sample_count] if field in REQUIRED_CACHE_FIELDS else value
            for field, value in cache.items()
        }
        for model, cache in caches.items()
    }
    validate_model_caches(sliced_caches)
    return sliced_caches

##### Returns highest probability of a sample 

In [6]:
def max_confidence(cache):
    return np.asarray(cache["probabilities"]).max(axis=1)

##### Returns combined feature table for confidence, entropy, margin

In [7]:
def extract_probability_features(probabilities):
    probabilities = np.asarray(probabilities)
    # if probabilities.ndim != 2 or probabilities.shape[1] < 2:
    #     raise ValueError("probabilities must have shape [sample_count, class_count >= 2]")
    confidence = probabilities.max(axis=1)
    entropy = -(probabilities * np.log(probabilities + 1e-12)).sum(axis=1)
    top_two = np.partition(probabilities, -2, axis=1)[:, -2:]
    margin = top_two.max(axis=1) - top_two.min(axis=1)
    return np.column_stack([confidence, entropy, margin]).astype(np.float32)

##### Returns # of classification that a model did

In [8]:
def model_counts(model_names, models=MODELS):
    return {model: int(np.count_nonzero(model_names == model)) for model in models}

##### Returns # of execution each model started

In [9]:
def counter_counts(counter, models):
    return {model: int(counter[model]) for model in models}

##### Returns latency info

In [10]:
def latency_metrics(latencies_ms):
    return {
        "mean_latency_ms": float(latencies_ms.mean()), "median_latency_ms": float(np.median(latencies_ms)), "p95_latency_ms": float(np.percentile(latencies_ms, 95))
    }

## Heavy-Model Router
#### Trains and evaluate the Random Forest that chooses between ResNet-50 and ResNet-152 when both early models are IDK.

##### Return's # of 0s (ResNet-50) and 1s (ResNet-152)

In [11]:
def heavy_route_counts(route_labels):
    return {
        model: int(np.count_nonzero(route_labels == HEAVY_ROUTE_LABEL_BY_MODEL[model]))
        for model in HEAVY_MODELS
    }

##### Creates training data for Random Forest to train on. Returns 'router_feature' (Sample features of ResNet-18 & ResNet-34), 'route_labels' (Target label 0 or 1 for router), 'included_uncertain_sample_count' (# of samples where ResNet-18 & ResNet-34 failed), 'excluded_sample_count (# of samples not eligible for heavy models)

In [12]:
def build_heavy_router_dataset(caches, confidence_threshold=DYNAMIC_CONFIDENCE_THRESHOLD, require_correct=False):
    validate_model_caches(caches)
    resnet18_features = extract_probability_features(caches[RESNET18]["probabilities"])
    resnet34_features = extract_probability_features(caches[RESNET34]["probabilities"])
    true_labels = caches[RESNET18]["labels"]

    both_early_models_are_uncertain = (
        (resnet18_features[:, 0] < confidence_threshold)
        & (resnet34_features[:, 0] < confidence_threshold)
    )
    resnet50_is_confident = max_confidence(caches[RESNET50]) >= confidence_threshold
    resnet152_is_confident = max_confidence(caches[RESNET152]) >= confidence_threshold

    if require_correct:
        resnet50_is_eligible = resnet50_is_confident & (
            caches[RESNET50]["predictions"] == true_labels
        )
        resnet152_is_eligible = resnet152_is_confident & (
            caches[RESNET152]["predictions"] == true_labels
        )
    else:
        resnet50_is_eligible = resnet50_is_confident
        resnet152_is_eligible = resnet152_is_confident

    included = both_early_models_are_uncertain & (
        resnet50_is_eligible | resnet152_is_eligible
    )
    router_features = np.column_stack(
        [resnet18_features[included], resnet34_features[included]]
    )
    route_labels = np.where(
        resnet50_is_eligible[included],
        HEAVY_ROUTE_LABEL_BY_MODEL[RESNET50],
        HEAVY_ROUTE_LABEL_BY_MODEL[RESNET152],
    ).astype(np.int64)

    included_uncertain_sample_count = int(both_early_models_are_uncertain.sum())
    excluded_sample_count = included_uncertain_sample_count - len(route_labels)
    return router_features, route_labels, included_uncertain_sample_count, excluded_sample_count

In [13]:
def train_heavy_router(training_cache_sets, confidence_threshold=DYNAMIC_CONFIDENCE_THRESHOLD, require_correct=False):
    feature_parts = []
    route_label_parts = []
    included_uncertain_sample_count = 0
    excluded_sample_count = 0

    for caches in training_cache_sets:
        features, labels, uncertain_count, excluded_count = build_heavy_router_dataset(
            caches, confidence_threshold, require_correct
        )
        included_uncertain_sample_count += uncertain_count
        excluded_sample_count += excluded_count
        if len(labels):
            feature_parts.append(features)
            route_label_parts.append(labels)

    if not route_label_parts:
        raise ValueError("No eligible samples were found for heavy-router training")

    training_features = np.concatenate(feature_parts)
    training_route_labels = np.concatenate(route_label_parts)
    route_counts = heavy_route_counts(training_route_labels)
    requirement = (
        "meets the threshold and predicts correctly"
        if require_correct
        else "meets the confidence threshold"
    )

    print("Heavy-router training samples:", len(training_route_labels))
    print("ResNet-50 route labels:", route_counts[RESNET50])
    print("ResNet-152 route labels:", route_counts[RESNET152])
    print(f"Excluded samples (neither heavy model {requirement}):", excluded_sample_count)
    print("Samples where both early models are below the threshold:", included_uncertain_sample_count)

    heavy_router = RandomForestClassifier(
        n_estimators=100,
        max_depth=6,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=42,
    )
    heavy_router.fit(training_features, training_route_labels)
    return heavy_router

In [14]:



def evaluate_heavy_router(
    caches,
    heavy_router,
    confidence_threshold=DYNAMIC_CONFIDENCE_THRESHOLD,
    require_correct=False,
):
    features, target_labels, uncertain_count, excluded_count = (
        build_heavy_router_dataset(caches, confidence_threshold, require_correct)
    )
    predicted_labels = (
        np.asarray(heavy_router.predict(features), dtype=np.int64)
        if len(target_labels)
        else np.empty(0, dtype=np.int64)
    )
    route_label_accuracy = (
        float((predicted_labels == target_labels).mean())
        if len(target_labels)
        else float("nan")
    )
    confusion_matrix = np.zeros((2, 2), dtype=np.int64)
    if len(target_labels):
        np.add.at(confusion_matrix, (target_labels, predicted_labels), 1)

    result = {
        "both_early_uncertain_sample_count": uncertain_count,
        "evaluated_sample_count": len(target_labels),
        "excluded_sample_count": excluded_count,
        "route_label_accuracy": route_label_accuracy,
        "target_route_count_by_model": heavy_route_counts(target_labels),
        "predicted_route_count_by_model": heavy_route_counts(predicted_labels),
        "confusion_matrix": confusion_matrix,
    }

    print("Samples where both early models are below the threshold:", uncertain_count)
    print("Samples with a heavy-route target:", len(target_labels))
    print("Excluded samples without an eligible heavy model:", excluded_count)
    print("Heavy-route label accuracy:", route_label_accuracy)
    print("Target route counts:", result["target_route_count_by_model"])
    print("Predicted route counts:", result["predicted_route_count_by_model"])
    print("Confusion matrix (rows=target, columns=predicted; ResNet-50 then ResNet-152):")
    print(confusion_matrix)
    return result

## Sequential Random Forest for Baseline (Ronit's Paper)

In [15]:
def train_sequential_skip_router(
    training_cache_sets,
    confidence_threshold=SEQUENTIAL_CONFIDENCE_THRESHOLD,
):
    feature_parts = []
    skip_label_parts = []

    for caches in training_cache_sets:
        validate_model_caches(caches)
        resnet18_is_uncertain = max_confidence(caches[RESNET18]) < confidence_threshold
        feature_parts.append(
            extract_probability_features(
                caches[RESNET18]["probabilities"][resnet18_is_uncertain]
            )
        )
        skip_label_parts.append(
            np.where(
                max_confidence(caches[RESNET34])[resnet18_is_uncertain]
                < confidence_threshold,
                SKIP_RESNET34_LABEL,
                USE_RESNET34_LABEL,
            )
        )

    skip_router = RandomForestClassifier(
        n_estimators=50,
        max_depth=4,
        min_samples_leaf=40,
        class_weight="balanced",
        random_state=42,
        n_jobs=1,
    )
    skip_router.fit(np.concatenate(feature_parts), np.concatenate(skip_label_parts))
    return skip_router


def run_sequential_baseline(caches, skip_router):
    sample_count = validate_model_caches(caches)
    true_labels = caches[RESNET18]["labels"]
    resnet18_confidence = max_confidence(caches[RESNET18])
    resnet34_confidence = max_confidence(caches[RESNET34])

    final_predictions = np.empty(sample_count, dtype=np.int64)
    final_prediction_models = np.empty(sample_count, dtype=object)
    latencies_ms = caches[RESNET18]["times_ms"].astype(float).copy()

    resnet18_supplies_prediction = (
        resnet18_confidence >= SEQUENTIAL_CONFIDENCE_THRESHOLD
    )
    routed_indices = np.flatnonzero(~resnet18_supplies_prediction)
    final_predictions[resnet18_supplies_prediction] = caches[RESNET18]["predictions"][
        resnet18_supplies_prediction
    ]
    final_prediction_models[resnet18_supplies_prediction] = RESNET18

    resnet34_is_skipped = np.zeros(sample_count, dtype=bool)
    if len(routed_indices):
        routing_start = time.perf_counter()
        skip_labels = skip_router.predict(
            extract_probability_features(caches[RESNET18]["probabilities"][routed_indices])
        )
        routing_time_ms = (time.perf_counter() - routing_start) * 1000.0
        latencies_ms[routed_indices] += routing_time_ms / len(routed_indices)
        resnet34_is_skipped[routed_indices] = skip_labels == SKIP_RESNET34_LABEL

    resnet34_is_executed = (~resnet18_supplies_prediction) & ~resnet34_is_skipped
    resnet34_supplies_prediction = (
        resnet34_is_executed
        & (resnet34_confidence >= SEQUENTIAL_CONFIDENCE_THRESHOLD)
    )
    resnet152_is_executed = resnet34_is_skipped | (
        resnet34_is_executed & ~resnet34_supplies_prediction
    )

    latencies_ms[resnet34_is_executed] += caches[RESNET34]["times_ms"][
        resnet34_is_executed
    ]
    latencies_ms[resnet152_is_executed] += caches[RESNET152]["times_ms"][
        resnet152_is_executed
    ]
    final_predictions[resnet34_supplies_prediction] = caches[RESNET34]["predictions"][
        resnet34_supplies_prediction
    ]
    final_predictions[resnet152_is_executed] = caches[RESNET152]["predictions"][
        resnet152_is_executed
    ]
    final_prediction_models[resnet34_supplies_prediction] = RESNET34
    final_prediction_models[resnet152_is_executed] = RESNET152

    total_serial_time_ms = float(latencies_ms.sum())
    return {
        "sample_count": sample_count,
        "accuracy": float((final_predictions == true_labels).mean()),
        "total_serial_time_ms": total_serial_time_ms,
        "throughput_fps": sample_count / (total_serial_time_ms / 1000.0),
        **latency_metrics(latencies_ms),
        "final_prediction_count_by_model": model_counts(final_prediction_models),
    }

## Parallel Worker Reuse Simulation

In [16]:
def simulate_parallel_scheduler(
    caches,
    heavy_router,
    confidence_threshold=DYNAMIC_CONFIDENCE_THRESHOLD,
    max_in_flight=None,
):
    sample_count = validate_model_caches(caches)
    if max_in_flight is not None:
        if not isinstance(max_in_flight, (int, np.integer)) or max_in_flight <= 0:
            raise ValueError("max_in_flight must be None or a positive integer")
        max_in_flight = int(max_in_flight)

    true_labels = caches[RESNET18]["labels"]
    resnet18_features = extract_probability_features(caches[RESNET18]["probabilities"])
    resnet34_features = extract_probability_features(caches[RESNET34]["probabilities"])
    early_confidence = {
        RESNET18: resnet18_features[:, 0],
        RESNET34: resnet34_features[:, 0],
    }
    predicted_heavy_routes = np.asarray(
        heavy_router.predict(np.column_stack([resnet18_features, resnet34_features])),
        dtype=np.int64,
    )

    start_times_ms = np.full(sample_count, np.nan)
    end_times_ms = np.full(sample_count, np.nan)
    final_predictions = np.full(sample_count, -1, dtype=np.int64)
    final_prediction_models = np.full(sample_count, None, dtype=object)
    early_model_completed = {
        model: np.zeros(sample_count, dtype=bool) for model in (RESNET18, RESNET34)
    }
    early_model_is_uncertain = {
        model: np.zeros(sample_count, dtype=bool) for model in (RESNET18, RESNET34)
    }
    heavy_job_is_queued = np.zeros(sample_count, dtype=bool)

    active_sample_by_model = {model: None for model in MODELS}
    queued_samples_by_model = {
        RESNET34: deque(),
        RESNET50: deque(),
        RESNET152: deque(),
    }
    completion_events = []
    event_sequence = 0
    next_sample_index = 0
    in_flight_count = 0
    completed_sample_count = 0
    simulation_time_ms = 0.0

    execution_count_by_model = Counter()
    busy_time_ms_by_model = Counter()
    canceled_queue_count_by_model = Counter()
    unused_completion_count_by_model = Counter()
    heavy_route_count_by_model = Counter()

    def start_job(model, sample_index, current_time_ms):
        nonlocal event_sequence
        duration_ms = float(caches[model]["times_ms"][sample_index])
        active_sample_by_model[model] = sample_index
        execution_count_by_model[model] += 1
        busy_time_ms_by_model[model] += duration_ms
        event_sequence += 1
        heapq.heappush(
            completion_events,
            (current_time_ms + duration_ms, event_sequence, model, sample_index),
        )

    def dispatch(model, current_time_ms):
        if active_sample_by_model[model] is not None:
            return
        while queued_samples_by_model[model]:
            sample_index = queued_samples_by_model[model].popleft()
            if final_prediction_models[sample_index] is not None:
                canceled_queue_count_by_model[model] += 1
                continue
            start_job(model, sample_index, current_time_ms)
            return

    def admit_sample(current_time_ms):
        nonlocal next_sample_index, in_flight_count
        if active_sample_by_model[RESNET18] is not None or next_sample_index >= sample_count:
            return
        if max_in_flight is not None and in_flight_count >= max_in_flight:
            return
        sample_index = next_sample_index
        next_sample_index += 1
        in_flight_count += 1
        start_times_ms[sample_index] = current_time_ms
        start_job(RESNET18, sample_index, current_time_ms)
        queued_samples_by_model[RESNET34].append(sample_index)

    def finalize(sample_index, model, current_time_ms):
        nonlocal in_flight_count, completed_sample_count
        if final_prediction_models[sample_index] is not None:
            return
        final_prediction_models[sample_index] = model
        final_predictions[sample_index] = caches[model]["predictions"][sample_index]
        end_times_ms[sample_index] = current_time_ms
        in_flight_count -= 1
        completed_sample_count += 1

    admit_sample(0.0)
    dispatch(RESNET34, 0.0)

    while completed_sample_count < sample_count or completion_events:
        if not completion_events:
            raise RuntimeError("Scheduler stopped with unfinished samples")

        simulation_time_ms = completion_events[0][0]
        simultaneous_completions = []
        while completion_events and completion_events[0][0] == simulation_time_ms:
            simultaneous_completions.append(heapq.heappop(completion_events))

        completed_models_by_sample = {}
        for _, _, model, sample_index in simultaneous_completions:
            if active_sample_by_model[model] != sample_index:
                raise RuntimeError("Unexpected worker completion")
            active_sample_by_model[model] = None

            if final_prediction_models[sample_index] is not None:
                unused_completion_count_by_model[model] += 1
                continue

            completed_models_by_sample.setdefault(sample_index, []).append(model)
            if model in early_confidence:
                early_model_completed[model][sample_index] = True
                early_model_is_uncertain[model][sample_index] = (
                    early_confidence[model][sample_index] < confidence_threshold
                )

        # Resolve all equal-time completions before dispatching new work.
        for sample_index, completed_models in completed_models_by_sample.items():
            completed_models = set(completed_models)
            if (
                RESNET18 in completed_models
                and not early_model_is_uncertain[RESNET18][sample_index]
            ):
                finalize(sample_index, RESNET18, simulation_time_ms)
                continue
            if (
                RESNET34 in completed_models
                and not early_model_is_uncertain[RESNET34][sample_index]
            ):
                finalize(sample_index, RESNET34, simulation_time_ms)
                continue
            if RESNET50 in completed_models:
                finalize(sample_index, RESNET50, simulation_time_ms)
                continue
            if RESNET152 in completed_models:
                finalize(sample_index, RESNET152, simulation_time_ms)
                continue

            both_early_models_are_uncertain = all(
                early_model_completed[model][sample_index]
                and early_model_is_uncertain[model][sample_index]
                for model in (RESNET18, RESNET34)
            )
            if both_early_models_are_uncertain and not heavy_job_is_queued[sample_index]:
                route_label = int(predicted_heavy_routes[sample_index])
                heavy_model = HEAVY_MODEL_BY_ROUTE_LABEL.get(route_label)
                if heavy_model is None:
                    raise ValueError("Heavy router must predict route label 0 or 1")
                heavy_job_is_queued[sample_index] = True
                heavy_route_count_by_model[heavy_model] += 1
                queued_samples_by_model[heavy_model].append(sample_index)

        admit_sample(simulation_time_ms)
        for model in (RESNET34, RESNET50, RESNET152):
            dispatch(model, simulation_time_ms)

    latencies_ms = end_times_ms - start_times_ms
    return {
        "sample_count": sample_count,
        "accuracy": float((final_predictions == true_labels).mean()),
        "simulation_makespan_ms": float(simulation_time_ms),
        "throughput_fps": sample_count / (simulation_time_ms / 1000.0),
        **latency_metrics(latencies_ms),
        "final_prediction_count_by_model": model_counts(final_prediction_models),
        "execution_count_by_model": counter_counts(execution_count_by_model, MODELS),
        "utilization_by_model": {
            model: float(busy_time_ms_by_model[model] / simulation_time_ms)
            for model in MODELS
        },
        "heavy_route_count_by_model": counter_counts(
            heavy_route_count_by_model, HEAVY_MODELS
        ),
        "canceled_queue_count_by_model": counter_counts(
            canceled_queue_count_by_model, MODELS
        ),
        "unused_completion_count_by_model": counter_counts(
            unused_completion_count_by_model, MODELS
        ),
    }

## Run the Cached-Output Experiment

In [17]:
def cache_paths(artifact_prefix):
    return {
        model: f"artifacts/{artifact_prefix}_{model}.npz"
        for model in MODELS
    }


training_cache_sets = [
    load_model_caches(cache_paths(prefix)) for prefix in ("matched", "top")
]
test_caches = load_model_caches(cache_paths("threshold07"))
max_in_flight_options = (None, 2, 3, 4, 8, 16, 32)

heavy_router = train_heavy_router(training_cache_sets)
heavy_router_evaluation = evaluate_heavy_router(test_caches, heavy_router)
sequential_skip_router = train_sequential_skip_router(training_cache_sets)
sequential_result = run_sequential_baseline(test_caches, sequential_skip_router)

parallel_results_by_max_in_flight = {}
for max_in_flight in max_in_flight_options:
    limit_label = "unlimited" if max_in_flight is None else max_in_flight
    print(f"Running parallel scheduler (max in-flight samples: {limit_label})")
    parallel_results_by_max_in_flight[max_in_flight] = simulate_parallel_scheduler(
        test_caches, heavy_router, max_in_flight=max_in_flight
    )

Heavy-router training samples: 1111
ResNet-50 route labels: 254
ResNet-152 route labels: 857
Excluded samples (neither heavy model meets the confidence threshold): 1084
Samples where both early models are below the threshold: 2195
Samples where both early models are below the threshold: 1010
Samples with a heavy-route target: 543
Excluded samples without an eligible heavy model: 467
Heavy-route label accuracy: 0.6850828729281768
Target route counts: {'resnet50': 145, 'resnet152': 398}
Predicted route counts: {'resnet50': 212, 'resnet152': 331}
Confusion matrix (rows=target, columns=predicted; ResNet-50 then ResNet-152):
[[ 93  52]
 [119 279]]
Running parallel scheduler (max in-flight samples: unlimited)
Running parallel scheduler (max in-flight samples: 2)
Running parallel scheduler (max in-flight samples: 3)
Running parallel scheduler (max in-flight samples: 4)
Running parallel scheduler (max in-flight samples: 8)
Running parallel scheduler (max in-flight samples: 16)
Running parallel

## Results from simulation

In [18]:
def max_in_flight_label(limit):
    return "unlimited" if limit is None else str(limit)


comparison_runs = [("Sequential RF baseline", sequential_result)]
comparison_runs += [
    (
        f"Parallel scheduler (max in-flight: {max_in_flight_label(limit)})",
        parallel_results_by_max_in_flight[limit],
    )
    for limit in max_in_flight_options
]

print(
    f"{'System':<49} {'Accuracy':>8} {'Throughput (FPS)':>16} "
    f"{'Mean latency (ms)':>18} {'Median latency (ms)':>20} {'P95 latency (ms)':>17}"
)
for system_name, result in comparison_runs:
    print(
        f"{system_name:<49} {result['accuracy']:>8.3f} "
        f"{result['throughput_fps']:>16.2f} {result['mean_latency_ms']:>18.2f} "
        f"{result['median_latency_ms']:>20.2f} {result['p95_latency_ms']:>17.2f}"
    )

scheduler_detail_labels = {
    "final_prediction_count_by_model": "Final predictions by model",
    "execution_count_by_model": "Executions started by model",
    "utilization_by_model": "Worker utilization by model",
    "heavy_route_count_by_model": "Heavy routes assigned by model",
    "canceled_queue_count_by_model": "Queued jobs canceled before execution",
    "unused_completion_count_by_model": (
        "Completed executions unused after earlier finalization"
    ),
}
for limit, result in parallel_results_by_max_in_flight.items():
    print(f"\nParallel details (max in-flight samples: {max_in_flight_label(limit)})")
    for result_key, label in scheduler_detail_labels.items():
        print(f"{label}: {result[result_key]}")

System                                            Accuracy Throughput (FPS)  Mean latency (ms)  Median latency (ms)  P95 latency (ms)
Sequential RF baseline                               0.789            57.94              17.26                26.65             31.92
Parallel scheduler (max in-flight: unlimited)        0.714           333.45               7.11                 2.98             31.99
Parallel scheduler (max in-flight: 2)                0.714           262.51               6.14                 2.98             30.84
Parallel scheduler (max in-flight: 3)                0.714           301.63               6.50                 2.98             31.22
Parallel scheduler (max in-flight: 4)                0.714           319.71               6.77                 2.98             31.82
Parallel scheduler (max in-flight: 8)                0.714           333.27               7.11                 2.98             32.02
Parallel scheduler (max in-flight: 16)               0.714    

## Bunch of images at once Comparison for fps


In [19]:
def print_run_summary(system_name, result, duration_key, duration_label):
    print(f"{system_name}:")
    print("Samples:", result["sample_count"])
    print("Accuracy:", result["accuracy"])
    print(f"{duration_label} (seconds):", result[duration_key] / 1000.0)
    print("Throughput (FPS):", result["throughput_fps"])
    print("Mean latency (ms):", result["mean_latency_ms"])


comparison_sample_count = 5000
max_in_flight = 3
comparison_caches = take_first_samples(test_caches, comparison_sample_count)
sequential_subset_result = run_sequential_baseline(
    comparison_caches, sequential_skip_router
)
parallel_subset_result = simulate_parallel_scheduler(
    comparison_caches, heavy_router, max_in_flight=max_in_flight
)

print_run_summary(
    "Sequential RF baseline",
    sequential_subset_result,
    "total_serial_time_ms",
    "Total serial inference time",
)
print()
print_run_summary(
    f"Parallel scheduler (max in-flight samples: {max_in_flight})",
    parallel_subset_result,
    "simulation_makespan_ms",
    "Simulation makespan",
)

Sequential RF baseline:
Samples: 5000
Accuracy: 0.7922
Total serial inference time (seconds): 87.76732782301951
Throughput (FPS): 56.96880745967756
Mean latency (ms): 17.553465564603904

Parallel scheduler (max in-flight samples: 3):
Samples: 5000
Accuracy: 0.7132
Simulation makespan (seconds): 16.713896990182548
Throughput (FPS): 299.15225652861886
Mean latency (ms): 6.525460368832318


## Real-Time Parallel Worker Test

In [20]:
import json
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

MAX_SAMPLES = 5000
BATCH_SIZE = 1
MAX_WORKERS = 4
SAVE_RESULTS = True

TEST_IMAGE_FOLDER = Path("data/imagenetv2/imagenetv2-threshold0.7-format-val")
RESULTS_PATH = Path("real_parallel_worker_results.json")
PREDICTIONS_PATH = Path("real_parallel_worker_predictions.npz")

LIGHT_MODELS = (RESNET18, RESNET34)
CONFIDENCE_THRESHOLD = DYNAMIC_CONFIDENCE_THRESHOLD

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("Device:", DEVICE)

Device: mps


In [21]:
preprocess = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)

full_test_dataset = datasets.ImageFolder(TEST_IMAGE_FOLDER, transform=preprocess)
# ImageFolder sorts class names as strings, so restore the numeric ImageNet labels.
full_test_dataset.samples = [
    (path, int(Path(path).parent.name)) for path, _ in full_test_dataset.samples
]
full_test_dataset.imgs = full_test_dataset.samples
full_test_dataset.targets = [label for _, label in full_test_dataset.samples]

real_sample_count = min(MAX_SAMPLES, len(full_test_dataset))
real_test_dataset = Subset(full_test_dataset, range(real_sample_count))
real_test_loader = DataLoader(
    real_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=DEVICE.type == "cuda",
)

model_specs = {
    RESNET18: (models.resnet18, models.ResNet18_Weights.DEFAULT),
    RESNET34: (models.resnet34, models.ResNet34_Weights.DEFAULT),
    RESNET50: (models.resnet50, models.ResNet50_Weights.DEFAULT),
    RESNET152: (models.resnet152, models.ResNet152_Weights.DEFAULT),
}

real_models = {}
for model_name in MODELS:
    model_factory, weights = model_specs[model_name]
    real_models[model_name] = model_factory(weights=weights).to(DEVICE).eval()

print("Loaded samples:", real_sample_count)
print("Loaded models:", ", ".join(MODELS))

Loaded samples: 5000
Loaded models: resnet18, resnet34, resnet50, resnet152


In [22]:
real_system_run_complete = False
real_system_results = None
real_final_predictions = None
real_chosen_models = None

def run_real_model(model_name, images):
    with torch.no_grad():
        logits = real_models[model_name](images)
        probabilities = torch.softmax(logits, dim=1)[0].detach().cpu().numpy()
    prediction = int(probabilities.argmax())
    confidence = float(probabilities.max())
    return probabilities, prediction, confidence


total_samples = len(real_test_dataset)
labels = np.full(total_samples, -1, dtype=np.int64)
final_predictions = np.full(total_samples, -1, dtype=np.int64)
chosen_models = np.full(total_samples, "", dtype="<U16")
latencies_ms = np.full(total_samples, np.nan, dtype=np.float64)
execution_count_by_model = Counter()

sample_states = {}
pending = {}
next_sample_index = 0
completed_sample_count = 0
loader_iterator = iter(real_test_loader)
run_start = time.perf_counter()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    while completed_sample_count < total_samples or pending:
        # Admit only when both light-model jobs fit in the worker pool.
        while (
            next_sample_index < total_samples
            and len(pending) <= MAX_WORKERS - len(LIGHT_MODELS)
        ):
            images, batch_labels = next(loader_iterator)
            sample_index = next_sample_index
            next_sample_index += 1

            images = images.to(DEVICE, non_blocking=DEVICE.type == "cuda")
            labels[sample_index] = int(batch_labels.item())
            sample_states[sample_index] = {
                "images": images,
                "start_time": time.perf_counter(),
                "results": {},
                "finished": False,
            }

            for model_name in LIGHT_MODELS:
                future = executor.submit(run_real_model, model_name, images)
                pending[future] = (sample_index, model_name)
                execution_count_by_model[model_name] += 1

        if not pending:
            if completed_sample_count != total_samples:
                raise RuntimeError("Scheduler stopped before all samples completed")
            break

        done, _ = wait(tuple(pending), return_when=FIRST_COMPLETED)
        affected_samples = set()

        for future in done:
            sample_index, model_name = pending.pop(future)
            probabilities, prediction, confidence = future.result()
            sample_states[sample_index]["results"][model_name] = {
                "probabilities": probabilities,
                "prediction": prediction,
                "confidence": confidence,
            }
            affected_samples.add(sample_index)

        decision_time = time.perf_counter()
        for sample_index in affected_samples:
            state = sample_states[sample_index]
            if state["finished"]:
                continue

            heavy_results = [
                model_name
                for model_name in HEAVY_MODELS
                if model_name in state["results"]
            ]
            if heavy_results:
                chosen_model = heavy_results[0]
                final_predictions[sample_index] = state["results"][chosen_model][
                    "prediction"
                ]
                chosen_models[sample_index] = chosen_model
                latencies_ms[sample_index] = (
                    decision_time - state["start_time"]
                ) * 1000.0
                state["finished"] = True
                state["images"] = None
                completed_sample_count += 1
                continue

            confident_light_models = [
                model_name
                for model_name in LIGHT_MODELS
                if model_name in state["results"]
                and state["results"][model_name]["confidence"]
                >= CONFIDENCE_THRESHOLD
            ]
            if confident_light_models:
                chosen_model = max(
                    confident_light_models,
                    key=lambda name: state["results"][name]["confidence"],
                )
                final_predictions[sample_index] = state["results"][chosen_model][
                    "prediction"
                ]
                chosen_models[sample_index] = chosen_model
                latencies_ms[sample_index] = (
                    decision_time - state["start_time"]
                ) * 1000.0
                state["finished"] = True
                state["images"] = None
                completed_sample_count += 1
                continue

            if all(model_name in state["results"] for model_name in LIGHT_MODELS):
                resnet18_features = extract_probability_features(
                    state["results"][RESNET18]["probabilities"][None, :]
                )
                resnet34_features = extract_probability_features(
                    state["results"][RESNET34]["probabilities"][None, :]
                )
                router_features = np.column_stack(
                    [resnet18_features, resnet34_features]
                )
                route_label = int(heavy_router.predict(router_features)[0])
                heavy_model = HEAVY_MODEL_BY_ROUTE_LABEL.get(route_label)
                if heavy_model is None:
                    raise ValueError("Heavy router must predict route label 0 or 1")

                future = executor.submit(
                    run_real_model, heavy_model, state["images"]
                )
                pending[future] = (sample_index, heavy_model)
                execution_count_by_model[heavy_model] += 1

total_wall_time_seconds = time.perf_counter() - run_start
correct_predictions = int(np.count_nonzero(final_predictions == labels))
final_prediction_count_by_model = {
    model_name: int(np.count_nonzero(chosen_models == model_name))
    for model_name in MODELS
}

real_final_predictions = final_predictions
real_chosen_models = chosen_models
real_system_results = {
    "total_samples": total_samples,
    "accuracy": correct_predictions / total_samples,
    "correct_predictions": correct_predictions,
    "total_wall_time_seconds": total_wall_time_seconds,
    "throughput_fps": total_samples / total_wall_time_seconds,
    "mean_latency_ms": float(latencies_ms.mean()),
    "p50_latency_ms": float(np.percentile(latencies_ms, 50)),
    "p95_latency_ms": float(np.percentile(latencies_ms, 95)),
    "final_prediction_count_by_model": final_prediction_count_by_model,
    "execution_count_by_model": {
        model_name: int(execution_count_by_model[model_name])
        for model_name in MODELS
    },
}
real_system_run_complete = True

In [23]:
if not real_system_run_complete:
    raise RuntimeError("The real-system run did not complete; saved files were not changed")

print("Real-System Parallel Worker Test")
print("Total samples:", real_system_results["total_samples"])
print("Accuracy:", round(real_system_results["accuracy"], 4))
print("Correct predictions:", real_system_results["correct_predictions"])
print("Total wall time (seconds):", round(real_system_results["total_wall_time_seconds"], 3))
print("Throughput (FPS):", round(real_system_results["throughput_fps"], 3))
print("Mean latency (ms):", round(real_system_results["mean_latency_ms"], 3))
print("P50 latency (ms):", round(real_system_results["p50_latency_ms"], 3))
print("P95 latency (ms):", round(real_system_results["p95_latency_ms"], 3))
print("Final prediction count by model:")
for model_name in MODELS:
    print(f"  {model_name}: {real_system_results['final_prediction_count_by_model'][model_name]}")
print("Execution count by model:")
for model_name in MODELS:
    print(f"  {model_name}: {real_system_results['execution_count_by_model'][model_name]}")

if SAVE_RESULTS:
    results_temp_path = RESULTS_PATH.with_name(f".{RESULTS_PATH.name}.tmp")
    predictions_temp_path = PREDICTIONS_PATH.with_name(
        f".{PREDICTIONS_PATH.stem}.tmp.npz"
    )

    results_temp_path.write_text(
        json.dumps(real_system_results, indent=2) + "\n",
        encoding="utf-8",
    )
    np.savez_compressed(
        predictions_temp_path,
        predictions=real_final_predictions,
        chosen_models=real_chosen_models,
    )

    results_temp_path.replace(RESULTS_PATH)
    predictions_temp_path.replace(PREDICTIONS_PATH)
    print("Saved:", RESULTS_PATH)
    print("Saved:", PREDICTIONS_PATH)
else:
    print("SAVE_RESULTS is False; no files were written.")

Real-System Parallel Worker Test
Total samples: 5000
Accuracy: 0.7372
Correct predictions: 3686
Total wall time (seconds): 157.714
Throughput (FPS): 31.703
Mean latency (ms): 46.915
P50 latency (ms): 30.268
P95 latency (ms): 233.518
Final prediction count by model:
  resnet18: 4082
  resnet34: 517
  resnet50: 142
  resnet152: 259
Execution count by model:
  resnet18: 5000
  resnet34: 5000
  resnet50: 142
  resnet152: 259
Saved: real_parallel_worker_results.json
Saved: real_parallel_worker_predictions.npz
